# Лабораторная работа №1

In [18]:
import re, os, json, math, random
from collections import Counter, defaultdict
import numpy as np

## Параметры

In [19]:
CORPORA = {
    "tolstoy": "data/tolstoy_anna_karenina.txt",
    "dostoevsky": "data/dostoevsky_prestuplenie_i_nakazanie.txt",
}
PRIMARY = None
PROC_DIR, WEIGHTS_DIR = "data/processed", "weights"
SENT_END = {".", "!", "?"}
os.makedirs(PROC_DIR, exist_ok=True); os.makedirs(WEIGHTS_DIR, exist_ok=True)

In [20]:
ALPHA = 0.1                                   # сглаживание Лапласа
ALPHAS_TABLE = [1, 0.1, 0.01]                 # значения alpha для таблицы перплексии
TEMPERATURE = 1.0                             # температура генерации
HOLDOUT = 0.10                                # доля отложенной части
START_WORDS = ["он", "она", "но", "когда", "что"]   # стартовые слова для генерации
GEN_LEN = 30                                  # длина генерации в словах
SENT_MIN, SENT_MAX = 8, 15                    # длина предложения для проверки правдоподобия
SEED = 42

## Корпус и токенизация
Текст приводится к нижнему регистру и режется регулярным выражением на слова. Из знаков препинания остаются только `.`, `!`, `?` как отдельные токены, остальные выбрасываются. Эти знаки нужны для границ предложений (генерация, проверка правдоподобия, Наивный Байес). Последние 10 % токенов откладываются и в обучении не участвуют.

In [21]:
TOKEN_RE = re.compile(r"[а-яё]+(?:-[а-яё]+)*|[.!?]+")      # слово (в т.ч. через дефис) или знак конца предложения

def read_text(path):
    raw = open(path, "rb").read()
    texts = [raw.decode(e, errors="ignore") for e in ("utf-8", "cp1251", "koi8-r")]
    return max(texts, key=lambda t: t.lower().count(" и "))

def tokenize(text):
    return [t[0] if t[0] in SENT_END else t for t in TOKEN_RE.findall(text.lower())]

def split_sentences(tokens):
    sents, cur = [], []
    for t in tokens:
        cur.append(t)
        if t in SENT_END:                         # знак конца закрывает предложение
            sents.append(cur); cur = []
    return sents + ([cur] if cur else [])

def detok(tokens):
    s = re.sub(r" ([.!?])", r"\1", " ".join(tokens))
    return re.sub(r"(^|[.!?] )([а-яё])", lambda m: m.group(1) + m.group(2).upper(), s)

def build_corpus(name, path):
    tokens = tokenize(read_text(path))
    words = [t for t in tokens if t not in SENT_END]
    cut = int(len(tokens) * (1 - HOLDOUT))        # граница: последние 10 % откладываем
    corpus = {"n_words": len(words), "n_tokens": len(tokens), "vocab": len(set(words)),
              "top20": Counter(words).most_common(20), "train": tokens[:cut], "test": tokens[cut:]}
    with open(f"{PROC_DIR}/{name}.json", "w", encoding="utf-8") as f:
        json.dump(corpus, f, ensure_ascii=False)  # сохраняем обработанный корпус отдельно
    return corpus

corpora = {n: build_corpus(n, p) for n, p in CORPORA.items() if os.path.exists(p)}   # отсутствующие файлы пропускаются
PRIMARY = PRIMARY or next(iter(corpora))

for name, c in corpora.items():
    print(f"\n\n{name}\n\n")
    print(f"слов: {c['n_words']:,} | токенов (со знаками .!?): {c['n_tokens']:,} | словарь: {c['vocab']:,}")
    print(f"обучение: {len(c['train']):,} токенов | отложено: {len(c['test']):,} токенов")
    print("топ-20:", ", ".join(f"{w} ({n})" for w, n in c["top20"]))
print("\n\nОсновной корпус:", PRIMARY)



tolstoy


слов: 268,558 | токенов (со знаками .!?): 289,938 | словарь: 33,009
обучение: 260,944 токенов | отложено: 28,994 токенов
топ-20: и (12916), не (6537), что (5778), в (5719), он (5551), на (3594), она (3434), с (3328), я (3212), как (2660), но (2582), его (2578), это (2224), к (1983), ее (1804), все (1671), было (1656), так (1415), сказал (1412), а (1391)


dostoevsky


слов: 184,424 | токенов (со знаками .!?): 201,613 | словарь: 28,766
обучение: 181,451 токенов | отложено: 20,162 токенов
топ-20: и (9194), в (4336), не (3933), что (3355), он (2932), на (2549), я (2453), с (2246), а (1878), как (1726), это (1447), его (1322), но (1213), так (1203), же (1178), да (1061), к (1027), вы (976), всё (973), она (917)


Основной корпус: tolstoy


## Счётчики и вероятности
Обучение - это подсчёт: `unigram[w]` (сколько раз встретилось слово) и `bigram[v][w]` (сколько раз `w` стоит сразу после `v`). Вероятность следующего слова - доля

$$ P(w\mid v)=\frac{c(v,w)}{c(v)} $$

где `c(v)` - число пар, начинающихся с `v`, строка вероятностей суммируется в 1. Со сглаживанием Лапласа

$$ P(w\mid v)=\frac{c(v,w)+\alpha}{c(v)+\alpha V} $$

`V` - размер словаря обучающей части.

In [22]:
def sample_word(counts, T, rng):                  # выбор следующего слова по счётчикам с температурой T
    words = list(counts)
    c = np.array([counts[w] for w in words], dtype=np.float64)
    p = c / c.sum()                               # P(w|v) = c(v,w) / c(v)
    q = (p / p.max()) ** (1.0 / T)                # p^(1/T)
    return words[rng.choice(len(words), p=q / q.sum())]

In [23]:
class BigramLM:
    def fit(self, tokens):
        self.unigram = Counter(tokens)                        # c(w)
        self.bigram = defaultdict(Counter)
        for v, w in zip(tokens, tokens[1:]):
            self.bigram[v][w] += 1                            # c(v,w)
        self.ctx = {v: sum(c.values()) for v, c in self.bigram.items()}   # c(v) = сумма c(v,w) по w
        self.V = len(self.unigram)                            # размер словаря
        return self

    def prob(self, v, w, alpha=0.0):              # P(w|v), alpha=0, без сглаживания
        c_vw = self.bigram.get(v, {}).get(w, 0)
        den = self.ctx.get(v, 0) + alpha * self.V
        return (c_vw + alpha) / den if den > 0 else 0.0

    def logprob(self, text, alpha=0.0):           # logP = ∑_t*logP(w_t | w_t-1)
        toks = tokenize(text) if isinstance(text, str) else text
        total = 0.0
        for v, w in zip(toks, toks[1:]):
            p = self.prob(v, w, alpha)
            if p == 0:
                return float("-inf")              # пара не встречалась: log 0
            total += math.log(p)
        return total

    def perplexity(self, tokens, alpha=0.0):      # exp(-(1/N) * logP), N = число пар
        return math.exp(-self.logprob(tokens, alpha) / (len(tokens) - 1))

    def generate(self, start, n=30, T=1.0, rng=None):
        rng = rng or np.random.default_rng(SEED)
        out, n_words = [start], 1
        while n_words < n:                        # пока не набрано n слов
            counts = self.bigram.get(out[-1]) or self.unigram   # у слова нет продолжений, берём униграммы
            out.append(sample_word(counts, T, rng))
            n_words += out[-1] not in SENT_END
        return out

    def save(self, path):                         # веса модели = счётчики
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"unigram": self.unigram, "bigram": self.bigram}, f, ensure_ascii=False)

models = {name: BigramLM().fit(c["train"]) for name, c in corpora.items()}   # обучение только на обучающей части
m = models[PRIMARY]

v = START_WORDS[0]
print(f"сумма P(w|'{v}') =", round(sum(m.prob(v, w) for w in m.bigram[v]), 12))
for w, c in m.bigram[v].most_common(5):
    print(f"P({w!r} | {v!r}) = c({v},{w}) / c({v}) = {c}/{m.ctx[v]} = {m.prob(v, w):.4f}")

for name, mod in models.items():
    mod.save(f"{WEIGHTS_DIR}/bigram_{name}.json")
print("веса сохранены в", WEIGHTS_DIR)

сумма P(w|'он') = 1.0
P('не' | 'он') = c(он,не) / c(он) = 400/5003 = 0.0800
P('.' | 'он') = c(он,.) / c(он) = 228/5003 = 0.0456
P('был' | 'он') = c(он,был) / c(он) = 165/5003 = 0.0330
P('и' | 'он') = c(он,и) / c(он) = 100/5003 = 0.0200
P('чувствовал' | 'он') = c(он,чувствовал) / c(он) = 96/5003 = 0.0192
веса сохранены в weights


## Генерация текста и температура

In [24]:
rng = np.random.default_rng(SEED)
for s in START_WORDS:                             # пять текстов по 30 слов, T = 1
    print(f"[{s}]", detok(m.generate(s, GEN_LEN, T=1.0, rng=rng)), "\n")

[он] Он обыкновенно жена помешает. Да складывая зонтик из пензенского губернского предводителя и запел искусный и еще какое испытывает ученик после но в приют с некоторым соображениям анна аркадьевна. Всю низость 

[она] Она очутилась. Потом втягиваешься. И долли. Мне и отдалась вся эта холодность взгляда. Но прежде чем ты говоришь унизительное. Вронский три месяца был лишен смертью. Если это теперь после их 

[но] Но совершенно невозможно потому что делать. Благодарим покорно оставить мужа она взглянула на костяную шишечку звонка в которой она была огромная как он но и страх чтобы вы думаете? Я 

[когда] Когда он не забудь главное дело с хрипом дышавший подле него все это центр на то что и все рассчитано и видел что надо научить меня. Это было давнишнее намерение 

[что] Что он не поведет. Левина не имею понятие бытия. Они сидели почти в своей девочке все разболтал. Я его к ней речи как он обнял подушку взлохмаченного петрицкого. Графиня лидия 



In [25]:
for T in (0.5, 1, 2):                             # одни и те же стартовые слова и seed
    rng = np.random.default_rng(SEED)
    print(f"T = {T}\n")
    for s in START_WORDS[:3]:
        print(f"[{s}]", detok(m.generate(s, GEN_LEN, T=T, rng=rng)))
    print()

print("T     доля уникальных слов   средний P(w|v) выбранных пар")     # метрики по длинной генерации (1000 слов)
for T in (0.5, 1, 2):
    g = m.generate(START_WORDS[0], 1000, T=T, rng=np.random.default_rng(SEED))
    words = [t for t in g if t not in SENT_END]
    print(f"{T:<5} {len(set(words)) / len(words):<22.3f} {np.mean([m.prob(v, w) for v, w in zip(g, g[1:])]):.4f}")

T = 0.5

[он] Он сам не будет так же мы все его. Она не любил и она была в этом. Ты не только что ему. Я не мог понять. Я не хочет. В
[она] Она была в том что он не мог бы ты не понимаю. В то что он не может быть сомненье. Я не могла быть и с его. Я не могла
[но] Но я не может быть это все равно. Он не мог бы не только не мог бы не может быть только что он не было в это? Сказал он не

T = 1

[он] Он обыкновенно жена помешает. Да складывая зонтик из пензенского губернского предводителя и запел искусный и еще какое испытывает ученик после но в приют с некоторым соображениям анна аркадьевна. Всю низость
[она] Она очутилась. Потом втягиваешься. И долли. Мне и отдалась вся эта холодность взгляда. Но прежде чем ты говоришь унизительное. Вронский три месяца был лишен смертью. Если это теперь после их
[но] Но совершенно невозможно потому что делать. Благодарим покорно оставить мужа она взглянула на костяную шишечку звонка в которой она была огромная как он но и страх чтобы вы думаете? Я

T = 2

In [26]:
print(f"TEMPERATURE = {TEMPERATURE}")
rng = np.random.default_rng(SEED)
for s in START_WORDS:
    print(f"[{s}]", detok(m.generate(s, GEN_LEN, T=TEMPERATURE, rng=rng)), "\n")

TEMPERATURE = 1.0
[он] Он обыкновенно жена помешает. Да складывая зонтик из пензенского губернского предводителя и запел искусный и еще какое испытывает ученик после но в приют с некоторым соображениям анна аркадьевна. Всю низость 

[она] Она очутилась. Потом втягиваешься. И долли. Мне и отдалась вся эта холодность взгляда. Но прежде чем ты говоришь унизительное. Вронский три месяца был лишен смертью. Если это теперь после их 

[но] Но совершенно невозможно потому что делать. Благодарим покорно оставить мужа она взглянула на костяную шишечку звонка в которой она была огромная как он но и страх чтобы вы думаете? Я 

[когда] Когда он не забудь главное дело с хрипом дышавший подле него все это центр на то что и все рассчитано и видел что надо научить меня. Это было давнишнее намерение 

[что] Что он не поведет. Левина не имею понятие бытия. Они сидели почти в своей девочке все разболтал. Я его к ней речи как он обнял подушку взлохмаченного петрицкого. Графиня лидия 



#### Что меняется при смене температуры.
- `T = 0.5`: распределение заостряется, модель почти всегда берёт самые частые продолжения. Текст однообразный, крутится вокруг частых слов («не», «я», «сам»): доля уникальных слов самая низкая.
- `T = 1`: выбор точно по обучающим долям.
- `T = 2`: распределение сглаживается, редкие слова выбираются чаще. Текст становится разнообразнее, но связность теряется.

## Правдоподобие текста и сглаживание Лапласа

Из отложенной части берётся связное предложение с `SENT_MIN..SENT_MAX` слов и те же слова вперемешк, в которых знак конца остаётся на месте. Считается $\log P=\sum_t \log P(w_t \mid w_{t-1})$, сначала без сглаживания.

In [27]:
test = corpora[PRIMARY]["test"]
sent = next(s for s in split_sentences(test) if s[-1] in SENT_END and SENT_MIN <= len(s) - 1 <= SENT_MAX)   # первое подходящее
words = sent[:-1]                                 # слова без финального знака
random.Random(SEED).shuffle(words)                # те же слова вперемешку
coh, shuf = sent, words + sent[-1:]

print("БЕЗ сглаживания (alpha = 0)\n")
for label, t in (("связное", coh), ("вперемешку", shuf)):
    unseen = sum(m.prob(v, w) == 0 for v, w in zip(t, t[1:]))      # пары, которых не было в обучении
    print(f"{label:11}: {detok(t)}")
    print(f"{'':11}  пар: {len(t) - 1}, не встречавшихся в обучении: {unseen}, logP = {m.logprob(t)}")

БЕЗ сглаживания (alpha = 0)

связное    : Сказал степан аркадьич довольный тем что будут читать и дадут ему немножко опомниться.
             пар: 13, не встречавшихся в обучении: 6, logP = -inf
вперемешку : Читать будут аркадьич дадут опомниться что немножко и довольный тем сказал степан ему.
             пар: 13, не встречавшихся в обучении: 9, logP = -inf


### Почему без сглаживания сравнение невозможно
Если в предложении есть хотя бы одна пара `(v, w)`, не встречавшаяся в обучающей части, то `c(v,w) = 0`, `P(w|v) = 0`, а $\log 0 = -\infty$. Вся сумма $\log P$ становится $-\infty$ независимо от остальных пар. Обычно редкие пары есть в обоих предложениях, в нашем случае, 6 из 13 у связного и 9 из 13 у перемешанного, поэтому оба результата равны $-\infty$, и сравнивать нечего. Даже если $-\infty$ получит только одно предложение, мы узнаем лишь "бесконечно хуже", но не "насколько хуже". Сглаживание Лапласа даёт редкой паре малую ненулевую вероятность, и логарифм становится конечным.

In [28]:
print(f"СО сглаживанием (alpha = {ALPHA})\n")
for a in sorted(set(ALPHAS_TABLE + [ALPHA]), reverse=True):
    lc, ls = m.logprob(coh, a), m.logprob(shuf, a)
    print(f"alpha={a:<5} logP(связное)={lc:9.2f}   logP(вперемешку)={ls:9.2f}   разрыв={lc - ls:7.2f}" + ("  <- основное" if a == ALPHA else ""))

СО сглаживанием (alpha = 0.1)

alpha=1     logP(связное)=  -115.54   logP(вперемешку)=  -124.47   разрыв=   8.93
alpha=0.1   logP(связное)=  -104.02   logP(вперемешку)=  -119.48   разрыв=  15.46  <- основное
alpha=0.01  logP(связное)=   -96.97   logP(вперемешку)=  -119.31   разрыв=  22.34


## Перплексия на отложенных 10 %

$$\mathrm{PPL}=\exp\Big(-\frac{1}{N}\sum_t \log P(w_t \mid w_{t-1})\Big)$$

где `N` - число пар отложенной части.

In [29]:
for name, c in corpora.items():
    print(f"\n{name}: отложено {len(c['test']):,} токенов")
    print(f"{'alpha':>8} | {'перплексия':>12}")
    for a in [0.0] + ALPHAS_TABLE:
        print(f"{a:>8} | {models[name].perplexity(c['test'], a):>12.1f}")


tolstoy: отложено 28,994 токенов
   alpha |   перплексия
     0.0 |          inf
       1 |       7913.2
     0.1 |       3533.0
    0.01 |       2347.1

dostoevsky: отложено 20,162 токенов
   alpha |   перплексия
     0.0 |          inf
       1 |      13888.4
     0.1 |      10028.6
    0.01 |       9827.9


### Вывод 
Чем меньше `alpha` из таблицы, тем ниже перплексия. лЛуший результат здесь показывает `alpha = 0.01`, худший `alpha = 1`. 

Причина в том, что к знаменателю добавляется `alpha * V`, а `V` составляет десятки тысяч, тогда как `c(v)` у большинства слов - это единицы или десятки. При `alpha = 1` основная масса вероятности уходит на редкие слова, строка `P(·|v)` размывается к равномерной `1/V`, и информация из счётчиков теряется. Слишком малое `alpha` также плохо отразится на результате. При нем, редкие пары получают почти нулевую вероятность и сильно штрафуются. 

## Триграммы с откатом на биграммы

Контекст из двух слов: если тройка `(u, v, w)` встречалась в обучении, то

\\[ P(w\mid u,v)=\frac{c(u,v,w)}{c(u,v)} \\]

иначе откат на сглаженную биграмму `P(w|v)` (с тем же `alpha`). Это простой откат без перенормировки. У первой пары текста контекста из двух слов нет, там используется биграмма (`u = None`), поэтому число предсказаний `N` такое же, как у биграммной модели и перплексии сравнимы.

In [30]:
class TrigramLM:
    def fit(self, tokens):
        self.bi = BigramLM().fit(tokens)                      # биграммная модель для отката
        self.tri = defaultdict(Counter)
        for u, v, w in zip(tokens, tokens[1:], tokens[2:]):
            self.tri[(u, v)][w] += 1                          # c(u,v,w)
        self.ctx2 = {k: sum(c.values()) for k, c in self.tri.items()}   # c(u,v)
        return self

    def prob(self, u, v, w, alpha=0.0):
        c = self.tri.get((u, v), {}).get(w, 0)
        if c > 0:
            return c / self.ctx2[(u, v)]                      # тройка была: c(u,v,w) / c(u,v)
        return self.bi.prob(v, w, alpha)                      # тройки не было: откат на сглаженную биграмму

    def perplexity(self, tokens, alpha=0.0):
        triples = zip([None] + tokens, tokens, tokens[1:])    # (u, v, w); у первой пары u = None, биграмма
        ps = [self.prob(u, v, w, alpha) for u, v, w in triples]
        if 0.0 in ps:
            return float("inf")                               # редкая пара без сглаживания: log 0
        return math.exp(-sum(math.log(p) for p in ps) / (len(tokens) - 1))

    def generate(self, start, n=30, T=1.0, rng=None):
        rng = rng or np.random.default_rng(SEED)
        out, n_words = [start], 1
        while n_words < n:
            u = out[-2] if len(out) > 1 else None
            counts = self.tri.get((u, out[-1])) or self.bi.bigram.get(out[-1]) or self.bi.unigram   # тройки -> биграммы -> униграммы
            out.append(sample_word(counts, T, rng))
            n_words += out[-1] not in SENT_END
        return out

tri_models = {name: TrigramLM().fit(c["train"]) for name, c in corpora.items()}   # обучение только на обучающей части
tm = tri_models[PRIMARY]

for name, c in corpora.items():                   # перплексия на отложенной части: биграммы и триграммы
    print(f"\n{name}: перплексия на отложенной части")
    print(f"{'alpha':>6} | {'биграммы':>10} | {'триграммы':>10}")
    for a in ALPHAS_TABLE:
        print(f"{a:>6} | {models[name].perplexity(c['test'], a):>10.1f} | {tri_models[name].perplexity(c['test'], a):>10.1f}")


tolstoy: перплексия на отложенной части
 alpha |   биграммы |  триграммы
     1 |     7913.2 |     3358.9
   0.1 |     3533.0 |     2047.0
  0.01 |     2347.1 |     1685.6

dostoevsky: перплексия на отложенной части
 alpha |   биграммы |  триграммы
     1 |    13888.4 |    10029.6
   0.1 |    10028.6 |     8003.0
  0.01 |     9827.9 |     8444.0


В таблице триграммная перплексия ниже биграммной при каждом `alpha`. Суть в том, что там, где тройка встречалась, модель опирается на более точный контекст. Но тройки из отложенной части в обучении встречаются редко, поэтому в большинстве строк разбора идёт откат на биграмму. Генерация триграммами связнее, потому что контекст длиннее, но она часто дословно повторяет фрагменты обучающего текста.

In [31]:
print(f"Генерация, T = {TEMPERATURE}: биграммы и триграммы (одни и те же стартовые слова и seed)")
for s in START_WORDS:
    print(f"[{s}] биграммы : {detok(m.generate(s, GEN_LEN, T=TEMPERATURE, rng=np.random.default_rng(SEED)))}")
    print(f"[{s}] триграммы: {detok(tm.generate(s, GEN_LEN, T=TEMPERATURE, rng=np.random.default_rng(SEED)))}\n")

Генерация, T = 1.0: биграммы и триграммы (одни и те же стартовые слова и seed)
[он] биграммы : Он обыкновенно жена помешает. Да складывая зонтик из пензенского губернского предводителя и запел искусный и еще какое испытывает ученик после но в приют с некоторым соображениям анна аркадьевна. Всю низость
[он] триграммы: Он обыкновенно приходил после блестящего и оживленного разговора. Заметив однако что же в убыток. Вот отлично! Стал просить васенька весловский подтрунивая над немцем. Обратился он к вошедшему лакею. Взяв правою

[она] биграммы : Она нравится. Положительно неприлична. Отъехав три господина в кити и разговаривали у одной и еще какое испытывает ученик после но в приют с некоторым соображениям анна аркадьевна. Всю низость. Я
[она] триграммы: Она нравится. На то что скучает но эта радостная улыбка сейчас же уехать с ним. Ты знаешь ее и не больной человек не мог не чувствовать себя недостойным ее еще

[но] биграммы : Но невольно склоняясь всем смотревшим на пюпитре требуют от н

In [32]:
# Как работает откат. Предложение из отложенной части, источник каждой вероятности
print(f"Разбор предложения (alpha = {ALPHA}): {detok(coh)}\n")
print(f"{'контекст':<26}{'слово':<14}{'источник':<20}{'c(u,v,w)':>9}{'P':>10}")
for u, v, w in zip([None] + coh, coh, coh[1:]):
    c3 = tm.tri.get((u, v), {}).get(w, 0)                     # c(u,v,w) = 0 -> был откат на биграмму
    src = 'триграмма' if c3 else ('откат на биграмму' if u else 'биграмма (u=None)')   # у первой пары контекста из 2 слов нет
    print(f"{(u + ' ' if u else '') + v:<26}{w:<14}{src:<20}{c3:>9}{tm.prob(u, v, w, ALPHA):>10.5f}")

# Пары и тройки слов из сгенерированного триграммной моделью текста
g = tm.generate(START_WORDS[0], GEN_LEN, T=TEMPERATURE, rng=np.random.default_rng(SEED))
print(f"\nТекст: {detok(g)}\n")
for v, w in list(zip(g, g[1:]))[:8]:
    print(f"  пара   ({v} -> {w}): c(v,w)={m.bigram.get(v, {}).get(w, 0)}")
for u, v, w in list(zip(g, g[1:], g[2:]))[:8]:
    print(f"  тройка ({u} {v} -> {w}): c(u,v,w)={tm.tri.get((u, v), {}).get(w, 0)}")

Разбор предложения (alpha = 0.1): Сказал степан аркадьич довольный тем что будут читать и дадут ему немножко опомниться.

контекст                  слово         источник             c(u,v,w)         P
сказал                    степан        биграмма (u=None)           0   0.02423
сказал степан             аркадьич      триграмма                 106   1.00000
степан аркадьич           довольный     откат на биграмму           0   0.00003
аркадьич довольный        тем           откат на биграмму           0   0.00003
довольный тем             что           откат на биграмму           0   0.02534
тем что                   будут         откат на биграмму           0   0.00013
что будут                 читать        откат на биграмму           0   0.00003
будут читать              и             откат на биграмму           0   0.00100
читать и                  дадут         откат на биграмму           0   0.00001
и дадут                   ему           откат на биграмму           0   0.0006

## Наивный Байес

Для каждого автора считаются частоты слов на обучающей части, затем предложения из отложенных частей классифицируются по автору:

\\[ \hat c=\arg\max_c\Big[\log P(c)+\sum_{w\in s}\log\frac{n_c(w)+\alpha}{N_c+\alpha V}\Big] \\]

`n_c(w)` - число вхождений слова у автора `c`, `N_c` - число слов автора, `V` - размер общего словаря, `P(c)` - доля обучающих предложений автора.

In [33]:
class NaiveBayes:
    def fit(self, class_sents):                   # {автор: [предложения как списки слов]}
        self.cnt = {c: Counter(w for s in S for w in s) for c, S in class_sents.items()}   # n_c(w)
        self.N = {c: sum(cnt.values()) for c, cnt in self.cnt.items()}                      # N_c
        self.vocab = set().union(*self.cnt.values())
        self.V = len(self.vocab)                  # размер общего словаря
        n_sents = sum(len(S) for S in class_sents.values())
        self.log_prior = {c: math.log(len(S) / n_sents) for c, S in class_sents.items()}   # log P(c)
        return self

    def predict(self, words, alpha):
        def score(c):                             # log P(c) + сумма log P(w|c) со сглаживанием
            return self.log_prior[c] + sum(math.log((self.cnt[c][w] + alpha) / (self.N[c] + alpha * self.V))
                                           for w in words if w in self.vocab)
        return max(self.cnt, key=score)           # автор с максимальным счётом

def word_sentences(tokens):                       # предложения как списки слов (без .!?)
    return [s for s in ([t for t in s if t not in SENT_END] for s in split_sentences(tokens)) if s]

if len(corpora) < 2:
    print("Нужно хотя-бы два корпуса произведений, а есть только один")
else:
    nb = NaiveBayes().fit({n: word_sentences(c["train"]) for n, c in corpora.items()})   # счётчики по обучающим частям
    classes = list(corpora)
    conf = {t: Counter() for t in classes}        # conf[истинный][предсказанный]
    buckets = {"1-4": [0, 0], "5-9": [0, 0], "10+": [0, 0]}   # [верно, всего] по длине предложения
    for true, c in corpora.items():
        for s in word_sentences(c["test"]):       # классифицируем предложения из отложенных частей
            pred = nb.predict(s, ALPHA)
            conf[true][pred] += 1
            b = "1-4" if len(s) <= 4 else "5-9" if len(s) <= 9 else "10+"
            buckets[b][0] += pred == true; buckets[b][1] += 1
    total = sum(sum(r.values()) for r in conf.values())
    print(f"\nalpha = {ALPHA} \n классы: {classes} \n предложений в отложенных частях: {total}\n")
    print(f"Общая точность: {sum(conf[c][c] for c in classes) / total:.1%}\n")
    for t in classes:
        n = sum(conf[t].values())
        print(f"  автор {t}: точность {conf[t][t] / n:.1%} ({n} предложений); ошибки -> " +
              ", ".join(f"{p}: {conf[t][p]}" for p in classes if p != t))
    print("\nТочность по длине предложения:", {b: f"{ok / n:.1%} (n={n})" for b, (ok, n) in buckets.items() if n})

    a_, b_ = classes[:2]                          # самые характерные слова: отношение вероятностей слова у двух авторов
    ratio = {w: math.log((nb.cnt[a_][w] + ALPHA) / (nb.N[a_] + ALPHA * nb.V))
                - math.log((nb.cnt[b_][w] + ALPHA) / (nb.N[b_] + ALPHA * nb.V))
             for w in nb.vocab if nb.cnt[a_][w] + nb.cnt[b_][w] >= 30}   # порог 30 только чтобы список был читаемым
    top = sorted(ratio, key=ratio.get)
    print(f"\nСлова, самые характерные для {b_}:", ", ".join(top[:10]))
    print(f"Слова, самые характерные для {a_}:", ", ".join(top[::-1][:10]))


alpha = 0.1 
 классы: ['tolstoy', 'dostoevsky'] 
 предложений в отложенных частях: 3548

Общая точность: 67.2%

  автор tolstoy: точность 81.9% (2116 предложений); ошибки -> dostoevsky: 384
  автор dostoevsky: точность 45.5% (1432 предложений); ошибки -> tolstoy: 780

Точность по длине предложения: {'1-4': '46.4% (n=997)', '5-9': '68.2% (n=830)', '10+': '78.7% (n=1721)'}

Слова, самые характерные для dostoevsky: всё, раскольников, соня, разумихин, свидригайлов, дуня, порфирий, раскольникова, пульхерия, родя
Слова, самые характерные для tolstoy: левин, кити, вронский, степан, аркадьич, александрович, долли, левина, вронского, левину


### Влияние написания "ё" и "е"
В текстах корпусов буква "ё" используется по-разному. В корпусе Толстого она заменена на "е" ("ее", "все" в топ-20 частых слов), а в корпусе Достоевского встречается ("всё" в топ-20). Поэтому слово "всё" оказывается первым в списке самых характерных слов Достоевского, а "все" учитывается как другое слово у Толстого. Часть точности классификатора объясняется различием в написании, то есть особенностями изданий, а не стиля авторов. Нормализация "ё" в "е" в работе не выполняется. В реальных текстах "ё" часто опускается, и единого правила нет. Результат Наивного Байеса (общая точность 67.2 %) нужно читать с учётом этого эффекта.